In [2]:
import numpy as np
from scipy.spatial import cKDTree

rng = np.random.default_rng(1)
R_OH, HALF = 0.9572, np.radians(104.52) / 2
H1 = R_OH * np.array([ np.sin(HALF), np.cos(HALF), 0])
H2 = R_OH * np.array([-np.sin(HALF), np.cos(HALF), 0])

def rand_rot():
    q = rng.normal(size=4); q /= np.linalg.norm(q)
    a, b, c, d = q
    return np.array([
        [a*a+b*b-c*c-d*d, 2*(b*c-a*d),     2*(b*d+a*c)],
        [2*(b*c+a*d),     a*a-b*b+c*c-d*d, 2*(c*d-a*b)],
        [2*(b*d-a*c),     2*(c*d+a*b),     a*a-b*b-c*c+d*d]])

lines = open("../PDE4B_monomer.pdb").readlines()

# collect all heavy atoms (protein, ligand, waters)
xyz, polar, is_wat = [], [], []
for l in lines:
    if l.startswith(("ATOM", "HETATM")):
        xyz.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
        el = (l[76:78].strip() or l[12:16].strip()[0]).upper()
        polar.append(el in ("N", "O"))
        is_wat.append(l[17:20] == "HOH")
xyz, polar = np.array(xyz), np.array(polar)
tree = cKDTree(xyz)

def score(h, own):
    s = 0.0
    for j in tree.query_ball_point(h, 3.0):
        if j == own: continue
        d = np.linalg.norm(xyz[j] - h)
        if polar[j]:
            s += 10*(1.7 - d) if d < 1.7 else (-1.0 if d <= 2.2 else 0)
        elif d < 2.2:
            s += 10*(2.2 - d)
    return s


log = open("build_waters.log", "w")

out, n = [], 0
idx = -1
for l in lines:
    if l.startswith(("ATOM", "HETATM")):
        idx += 1
    if l.startswith("HETATM") and l[17:20] == "HOH":
        o = xyz[idx]
        best, best_s = None, 1e9
        for _ in range(5000):
            R = rand_rot()
            h1, h2 = o + R @ H1, o + R @ H2
            s = score(h1, idx) + score(h2, idx)
            if s < best_s: best, best_s = (h1, h2), s
        chain, resid = l[21], l[22:26]
        for name, p, el in (("OW", o, "O"), ("HW1", best[0], "H"), ("HW2", best[1], "H")):
            n += 1
            out.append(f"HETATM{n:5d} {name:<4s} HOH {chain}{resid}    "
                       f"{p[0]:8.3f}{p[1]:8.3f}{p[2]:8.3f}  1.00  0.00          {el:>2s}\n")
        print(f"water {resid.strip()}: score {best_s:.2f}")
        log.write(f"water {resid.strip()}: score {best_s:.2f}\n")
    else:
        pass

open("PDE4B_waters.pdb", "w").writelines(out)
log.close()

water 959: score -1.00
water 967: score -1.60
water 994: score 7.07
water 1015: score 0.00
water 1108: score -2.00
water 1118: score 0.00
